# PriceIQ — Phase 2 Final Notebook

Track B (Claude Agent SDK) · Sonnet Planner + Haiku Executor + 5 typed tools.

**Run order**: cells 1 → 10. Cell 9 (full 50-case eval) is optional and costs ~$2.57 — comment it out if you only want the demo.

**Secrets**: Set `ANTHROPIC_API_KEY`, `KAGGLE_API_TOKEN`, `OPENWEATHER_API_KEY` in Colab → Secrets panel before running.

Source: built from local priceiq_*.py via `build_ipynb.py`. Re-run that script after any module change.


## Cell 1 — Secrets + dependencies

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY']   = userdata.get('ANTHROPIC_API_KEY')
    os.environ['KAGGLE_API_TOKEN']    = userdata.get('KAGGLE_API_TOKEN')
    os.environ['OPENWEATHER_API_KEY'] = userdata.get('OPENWEATHER_API_KEY')
except ImportError:
    # Local run — assume env vars already set
    pass

!pip install -q anthropic kagglehub statsmodels plotly


## Cells 2–7 — Module sources via `%%writefile`

Each cell writes one module to the runtime filesystem. Run in order so imports resolve.

### Cell 2 — Tool 1 (`priceiq_data.py`) · Olist SQLite SQL

In [ ]:
%%writefile priceiq_data.py
"""PriceIQ Tool 1 — query_sales_data over Olist SQLite.

Data: Olist Brazilian E-Commerce dataset (Kaggle slug
`terencicp/e-commerce-dataset-by-olist-as-an-sqlite-database`).
9 tables (orders, order_items, products, customers, sellers, payments,
reviews, geolocation, category-name translation) · ~100K orders
2016-10 to 2018-08 · 71 product categories (we filter to `delivered`
status only, ~3% exclusion).

Public API:
    list_categories(min_orders=100) -> list[{pt, en, n_orders}]
    query_sales_data(category, start_date=None, end_date=None) -> dict
        Returns price stats + monthly_panel (used by Tools 2/3/5).

Token (KAGGLE_API_TOKEN, format `KGAT_...`):
    Colab → userdata.get('KAGGLE_API_TOKEN') (set in Secrets panel)
    Local → export KAGGLE_API_TOKEN=KGAT_xxx
"""

import os
import sqlite3
from typing import Optional

_DATASET = "terencicp/e-commerce-dataset-by-olist-as-an-sqlite-database"

# 模块级缓存（避免重复下载 / 重复 JOIN translation 表）
_db_path = None
_trans = None


def _ensure_db() -> str:
    """Lazy-download Olist SQLite (~110MB) and cache the path module-globally.

    First call: kagglehub fetches to ~/.cache/kagglehub (Colab) or local cache.
    Subsequent calls within the same process: returns cached `_db_path` for free.
    Cross-process caching is handled by kagglehub itself.
    """
    global _db_path
    if _db_path and os.path.exists(_db_path):
        return _db_path

    import kagglehub
    folder = kagglehub.dataset_download(_DATASET)
    for f in os.listdir(folder):
        if f.endswith(".sqlite"):
            _db_path = os.path.join(folder, f)
            return _db_path
    raise FileNotFoundError(f"No .sqlite file in {folder}")


def _translation() -> dict:
    """Portuguese → English category mapping (71 entries from Olist).

    Cached module-globally on first call. The translation table is small (71 rows)
    and immutable, so we hold the entire dict in memory rather than re-querying.
    """
    global _trans
    if _trans is not None:
        return _trans
    with sqlite3.connect(_ensure_db()) as conn:
        rows = conn.execute(
            "SELECT product_category_name, product_category_name_english "
            "FROM product_category_name_translation"
        ).fetchall()
    _trans = dict(rows)
    return _trans


def list_categories(min_orders: int = 100) -> list:
    """List Olist categories with ≥ min_orders delivered orders.

    Default min_orders=100 returns ~50/71 categories (excludes long-tail
    categories with too few orders for any meaningful elasticity estimate).
    Pass `min_orders=500` to get the top ~10 high-volume categories only.

    Returns: [{"pt": ..., "en": ..., "n_orders": ...}, ...] ranked desc by n_orders.
    """
    # Categories ranked by delivered-order volume; only categories with
    # ≥ min_orders are returned (filters noise from low-volume tail).
    sql = """
        SELECT p.product_category_name AS pt, COUNT(*) AS n
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE p.product_category_name IS NOT NULL
          AND o.order_status = 'delivered'
        GROUP BY p.product_category_name
        HAVING n >= ?
        ORDER BY n DESC
    """
    with sqlite3.connect(_ensure_db()) as conn:
        rows = conn.execute(sql, (min_orders,)).fetchall()
    trans = _translation()
    return [{"pt": pt, "en": trans.get(pt, pt), "n_orders": n} for pt, n in rows]


def query_sales_data(category: str,
                     start_date: Optional[str] = None,
                     end_date: Optional[str] = None) -> dict:
    """查询某品类的销售数据。

    Args:
        category: Olist 葡语品类名（如 'esporte_lazer'）
        start_date: ISO 日期 'YYYY-MM-DD'，None = 不限
        end_date: 同上

    Returns:
        dict 含 summary 统计 + monthly_panel（给 Tool 2 OLS 用）。
        无数据时 found=False。
    """
    # Only count `delivered` orders (Olist statuses include canceled,
    # unavailable, invoiced, shipped, etc.). For elasticity we want fulfilled
    # demand at the price point, not order intent. Excludes ~3% of records.
    where = ["p.product_category_name = ?", "o.order_status = 'delivered'"]
    params = [category]
    if start_date:
        where.append("o.order_purchase_timestamp >= ?")
        params.append(start_date)
    if end_date:
        where.append("o.order_purchase_timestamp <= ?")
        params.append(end_date)
    where_sql = " AND ".join(where)

    # 1) Aggregate summary (counts, price min/max/mean, freight mean, date range)
    summary_sql = f"""
        SELECT
            COUNT(*) AS n_orders,
            COUNT(DISTINCT o.order_id) AS n_unique_orders,
            MIN(oi.price) AS p_min,
            MAX(oi.price) AS p_max,
            AVG(oi.price) AS p_mean,
            AVG(oi.freight_value) AS f_mean,
            MIN(o.order_purchase_timestamp) AS d_min,
            MAX(o.order_purchase_timestamp) AS d_max
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE {where_sql}
    """

    # 2) Monthly panel (input to Tool 2 OLS + Tool 3 seasonality)
    panel_sql = f"""
        SELECT
            strftime('%Y-%m', o.order_purchase_timestamp) AS month,
            COUNT(*) AS n_orders,
            AVG(oi.price) AS avg_price,
            AVG(oi.freight_value) AS avg_freight,
            AVG(COALESCE(op.payment_installments, 1.0)) AS avg_installments
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN products p ON oi.product_id = p.product_id
        LEFT JOIN order_payments op ON oi.order_id = op.order_id
        WHERE {where_sql}
        GROUP BY month
        ORDER BY month
    """

    # 3) Sorted prices for median (Python computes median from this list)
    median_sql = f"""
        SELECT oi.price FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE {where_sql}
        ORDER BY oi.price
    """

    with sqlite3.connect(_ensure_db()) as conn:
        conn.row_factory = sqlite3.Row
        s = conn.execute(summary_sql, params).fetchone()
        if not s or s['n_orders'] == 0:
            return {"category": category, "found": False,
                    "message": f"No delivered orders for '{category}'"}
        panel = [dict(r) for r in conn.execute(panel_sql, params).fetchall()]
        prices = [r[0] for r in conn.execute(median_sql, params).fetchall()]

    median = prices[len(prices) // 2] if prices else 0.0

    return {
        "category": category,
        "category_english": _translation().get(category, category),
        "found": True,
        "n_orders": s['n_orders'],
        "n_unique_orders": s['n_unique_orders'],
        "date_range": [s['d_min'][:10], s['d_max'][:10]],
        "price_stats": {
            "min": round(s['p_min'], 2),
            "max": round(s['p_max'], 2),
            "mean": round(s['p_mean'], 2),
            "median": round(median, 2),
        },
        "freight_mean": round(s['f_mean'], 2),
        "monthly_panel": [
            {
                "month": r['month'],
                "n_orders": r['n_orders'],
                "avg_price": round(r['avg_price'], 2),
                "avg_freight": round(r['avg_freight'], 2),
                "avg_installments": round(r['avg_installments'], 2),
            } for r in panel
        ],
    }


if __name__ == "__main__":
    if not os.environ.get("KAGGLE_API_TOKEN"):
        print("⚠️  Set KAGGLE_API_TOKEN env var to run.")
        exit(0)

    print("\n── Top 10 categories ──")
    cats = list_categories(min_orders=500)
    for c in cats[:10]:
        print(f"  {c['pt']:35s} ({c['en']:30s}) {c['n_orders']:>6,}")

    print("\n── query_sales_data('esporte_lazer') ──")
    r = query_sales_data('esporte_lazer')
    for k, v in r.items():
        if k == 'monthly_panel':
            print(f"  {k}: {len(v)} months")
            print(f"    first: {v[0] if v else None}")
            print(f"    last:  {v[-1] if v else None}")
        else:
            print(f"  {k}: {v}")

    print("\n── query_sales_data('xyz_does_not_exist') ──")
    print(query_sales_data('xyz_does_not_exist'))


### Cell 3 — Tool 2 (`priceiq_elasticity.py`) · log-log OLS + multicollinearity

In [ ]:
%%writefile priceiq_elasticity.py
"""PriceIQ Tool 2 — calculate_price_elasticity (v2, log-log OLS).

Model:
  Naive:      ln(Q_t) = α + β·ln(P_t) + ε_t
  Controlled: ln(Q_t) = α + β·ln(P_t) + γ·freight_t + ε_t

Multicollinearity diagnostic (response to instructor comment #1):
  unstable := sign-flip(β_naive, β_ctrl)  OR  |Δβ| > 1.0
  if unstable: fall back to naive β + raise multicollinearity_warning

PVC v1 → v2: v1 used 7 predictors (ln_p + freight + installments + 3 quarter
dummies + const) on n=21 months → low df + collinearity → β flipped sign.
v2 collapses to 1 control (`avg_freight`) and adds the diagnostic.

Public API:
    calculate_price_elasticity(category, monthly_panel=None,
                               start_date=None, end_date=None) -> dict
        Includes: elasticity_beta, ci_95, p_value, multicollinearity_warning,
        recommended_source, naive_beta, controlled_beta, causal_caveat (verbatim).
"""

import os
from typing import Optional


def calculate_price_elasticity(
    category: str,
    monthly_panel: Optional[list] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> dict:
    """估计价格弹性 β，附 multicollinearity 诊断 + 推荐 β。"""
    if monthly_panel is None:
        from priceiq_data import query_sales_data
        d = query_sales_data(category, start_date, end_date)
        if not d.get("found"):
            return {"category": category, "success": False,
                    "message": d.get("message", "No data")}
        monthly_panel = d["monthly_panel"]

    # Need ≥8 monthly observations: controlled OLS uses 3 parameters (const +
    # ln_p + freight), leaving ≥5 residual degrees of freedom for stable CIs.
    # Below that, t-stats become meaningless even when the point estimate fits.
    if len(monthly_panel) < 8:
        return {"category": category, "success": False,
                "message": f"Insufficient panel: {len(monthly_panel)} months (need >= 8)"}

    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    df = pd.DataFrame(monthly_panel)
    df = df[(df["avg_price"] > 0) & (df["n_orders"] > 0)].copy()
    df["ln_q"] = np.log(df["n_orders"])
    df["ln_p"] = np.log(df["avg_price"])

    # ── 模型 A: naive log-log ─────────────────────────────
    X_simple = sm.add_constant(df[["ln_p"]].astype(float))
    m_simple = sm.OLS(df["ln_q"].values, X_simple).fit()

    # ── 模型 B: 单控制 (freight) ──────────────────────────
    # NOTE: v1 also included `avg_installments` and 3 quarter dummies → 7
    # predictors with n=21 hit multicollinearity (F-02). v2 reduced to a single
    # control (`avg_freight`) — captures the most-correlated confounder while
    # leaving plenty of residual df. See PVC_Log.md for the full v1→v2 trace.
    X_full = sm.add_constant(df[["ln_p", "avg_freight"]].astype(float))
    m_full = sm.OLS(df["ln_q"].values, X_full).fit()

    beta_naive = float(m_simple.params["ln_p"])
    beta_ctrl = float(m_full.params["ln_p"])

    # ── Multicollinearity / instability diagnostic ───────────
    # When `freight_value` is highly correlated with `ln_p` (often true in Olist —
    # higher-priced categories tend to have higher freight), the controlled OLS
    # can't cleanly separate effects. Symptoms: β flips sign, or |Δβ| explodes.
    # In those cases we honestly report the simpler naive estimate + a warning,
    # rather than a "controlled-but-wrong" number.
    sign_flipped = (beta_naive < 0) != (beta_ctrl < 0)
    big_jump = abs(beta_ctrl - beta_naive) > 1.0
    unstable = sign_flipped or big_jump

    if unstable:
        # Fallback to naive: less precision, but no spurious controlled effect.
        m = m_simple
        recommended_beta = beta_naive
        source = "naive (controlled model unstable: sign-flip or |Δβ|>1.0)"
    else:
        m = m_full
        recommended_beta = beta_ctrl
        source = "controlled with avg_freight"

    se = float(m.bse["ln_p"])
    ci_lo, ci_hi = m.conf_int(alpha=0.05).loc["ln_p"]
    p_val = float(m.pvalues["ln_p"])

    abs_beta = abs(recommended_beta)
    label = ("highly elastic" if abs_beta > 1.5 else
            "elastic" if abs_beta > 1.0 else
            "inelastic" if abs_beta > 0.5 else
            "very inelastic")

    return {
        "category": category,
        "success": True,
        "n_observations": int(len(df)),
        "elasticity_beta": round(recommended_beta, 3),
        "recommended_source": source,
        "naive_beta": round(beta_naive, 3),
        "controlled_beta": round(beta_ctrl, 3),
        "controls_changed_beta_by": round(beta_ctrl - beta_naive, 3),
        "multicollinearity_warning": unstable,
        "std_error": round(se, 3),
        "ci_95": [round(float(ci_lo), 3), round(float(ci_hi), 3)],
        "p_value": round(p_val, 4),
        "significance": "significant (p<0.05)" if p_val < 0.05 else "not significant",
        "r_squared": round(float(m.rsquared), 3),
        "elasticity_label": label,
        "control_variables": ["avg_freight"] if not unstable else [],
        "causal_caveat": (
            "ASSOCIATIONAL ONLY — not causal. Historical price variation in Olist "
            "data is confounded by promotions, freight policy changes, seasonality, "
            "and supply shocks we cannot fully observe. This beta reflects "
            "price-quantity correlation under the chosen control set, but does NOT "
            "prove that lowering price by 10%% will causally lift quantity by |beta|*10%%. "
            "Use as a directional indicator for pricing decisions, not as a causal estimate. "
            "A controlled A/B pricing experiment would be required for causal inference."
        ),
    }


### Cell 4 — Tool 3 (`priceiq_demand.py`) · BR holidays + seasonality

In [ ]:
%%writefile priceiq_demand.py
"""PriceIQ Tool 3 — get_demand_signals (BR holidays + Olist seasonality).

Formula (response to instructor comment #2):
    demand_multiplier = α_h * holiday_mult + α_s * seasonality_mult
    default α_h=0.4, α_s=0.6  (long-run seasonality weighted higher)

Holiday: 8 Brazilian holidays ±14 days, exponential decay (half-life 7 days).
Seasonality: month_avg / annual_avg from Olist delivered orders.

Public API:
    get_demand_signals(category, country='BR', today=None) -> dict
        Returns demand_multiplier + holiday + seasonality + sensitivity_analysis
        across α_h ∈ {0.2, 0.4, 0.6}.

Why not pytrends: unstable on Colab + multipliers were not interpretable
(instructor critique #2). Replaced with explicit α-weighted formula.
"""

import sqlite3
from datetime import date
from typing import Optional


# ── 巴西节假日 + 经验性影响倍数 ──────────────────────────────
# raw_impact: 该日期对 e-commerce 销量的相对影响（基于 Olist 数据观察）
# Hardcoded for 2026 (the submission year). For 2027+ regenerate dates from a
# BR holiday calendar source; impacts are stable. Phase 4 todo if course extends.
_BR_HOLIDAYS_2026 = [
    ("Carnival",        date(2026,  2, 17), -0.20),
    ("Easter",          date(2026,  4,  5), +0.10),
    ("Mothers Day",     date(2026,  5, 10), +0.15),
    ("Valentines BR",   date(2026,  6, 12), +0.15),
    ("Fathers Day",     date(2026,  8,  9), +0.10),
    ("Childrens Day",   date(2026, 10, 12), +0.25),  # major BR gift-giving holiday
    ("Black Friday",    date(2026, 11, 27), +0.30),
    ("Christmas",       date(2026, 12, 25), +0.30),
]

_HOLIDAY_RADIUS = 14   # ±14 days: covers consumer pre-shopping + post-clearance windows
_HOLIDAY_DECAY = 0.5   # 7-day half-life: matches typical promo cycle decay observed
                       # in retail panels (full strength on day, ~half a week out, ~quarter at radius edge)


def _holiday_signal(today: Optional[date] = None) -> dict:
    """返回距今最近节假日及其衰减后影响倍数。"""
    if today is None:
        today = date.today()

    nearest = None
    nearest_days = None
    raw = 0.0
    for name, d, impact in _BR_HOLIDAYS_2026:
        days = (d - today).days
        if -_HOLIDAY_RADIUS <= days <= _HOLIDAY_RADIUS:
            if nearest is None or abs(days) < abs(nearest_days):
                nearest, nearest_days, raw = name, days, impact

    if nearest is None:
        return {"nearest_holiday": None, "days_to_holiday": None,
                "holiday_multiplier": 1.0,
                "rationale": "No major BR holiday within +/-14 days"}

    decay = _HOLIDAY_DECAY ** (abs(nearest_days) / 7)
    mult = 1.0 + raw * decay
    # days_to_holiday: positive = future, negative = past, 0 = today
    when = (f"{nearest_days} days before {nearest}" if nearest_days > 0 else
            f"{abs(nearest_days)} days after {nearest}" if nearest_days < 0 else
            f"today is {nearest}")
    return {
        "nearest_holiday": nearest,
        "days_to_holiday": nearest_days,
        "holiday_multiplier": round(mult, 3),
        "rationale": f"{when} (raw impact {raw:+.0%}, decayed to {(mult-1)*100:+.1f}%)",
    }


def _seasonality_signal(category: str, ref_month: Optional[int] = None) -> dict:
    """品类的月度季节性 multiplier（month_avg / annual_avg）。"""
    if ref_month is None:
        ref_month = date.today().month

    # Reuse Tool 1's cached SQLite handle via priceiq_data._ensure_db
    from priceiq_data import _ensure_db
    with sqlite3.connect(_ensure_db()) as conn:
        # Aggregate delivered orders per calendar month (1-12) for this category.
        # Cross-year months are summed together (Jan 2017 + Jan 2018 = "month 1")
        # so the resulting pattern represents seasonal, not annual, variation.
        rows = conn.execute("""
            SELECT
                CAST(strftime('%m', o.order_purchase_timestamp) AS INTEGER) AS mo,
                COUNT(*) AS n
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            JOIN products p ON oi.product_id = p.product_id
            WHERE p.product_category_name = ?
              AND o.order_status = 'delivered'
            GROUP BY mo
            ORDER BY mo
        """, (category,)).fetchall()

    # Need ≥6 distinct months to give "seasonality" any meaning. Below that the
    # signal is dominated by single-month noise. Fall back to neutral 1.0.
    if len(rows) < 6:
        return {"seasonality_multiplier": 1.0, "monthly_pattern": {},
                "rationale": "Insufficient monthly data for seasonality"}

    # Multiplier = month_orders / mean_monthly_orders. 1.20 means this month
    # historically gets 20% more orders than the mean across observed months
    # (not strictly annual — sparse categories may miss some calendar months,
    # though the n<6 guard above filters out the most degenerate cases).
    # Months absent from `rows` (zero orders) get multiplier 1.0 via .get(),
    # which is conservative — they don't penalize the forecast.
    monthly = {mo: n for mo, n in rows}
    avg = sum(monthly.values()) / len(monthly)
    pattern = {mo: round(n / avg, 3) for mo, n in monthly.items()}
    mult = pattern.get(ref_month, 1.0)

    return {
        "seasonality_multiplier": mult,
        "ref_month": ref_month,
        "monthly_pattern": pattern,
        "n_months_observed": len(monthly),
        "rationale": (f"Month {ref_month} historically {(mult-1)*100:+.1f}% "
                      f"vs avg across {len(monthly)} observed months"),
    }


def get_demand_signals(category: str, country: str = "BR",
                       today: Optional[date] = None) -> dict:
    """组合 holiday + seasonality 给综合 demand multiplier。

    显式公式：
        demand_multiplier = α_h * holiday_mult + α_s * seasonality_mult
        默认 α_h = 0.4, α_s = 0.6（季节性长期更稳定，给更高权重）

    输出 sensitivity_analysis 字段，便于报告讨论权重选择。
    """
    if country != "BR":
        return {"category": category, "country": country, "applicable": False,
                "demand_multiplier": 1.0,
                "message": "Only BR supported in MVP"}

    h = _holiday_signal(today)
    s = _seasonality_signal(category)

    # α_s > α_h: long-run seasonality (24 months of category data) is more
    # statistically reliable than holiday proximity (single calendar event).
    # Sensitivity is reported below across α_h ∈ {0.2, 0.4, 0.6} so reviewers
    # can see how much these specific weights matter.
    ALPHA_H = 0.4
    ALPHA_S = 0.6
    demand_mult = ALPHA_H * h["holiday_multiplier"] + ALPHA_S * s["seasonality_multiplier"]
    demand_mult = max(0.5, min(1.8, demand_mult))   # clamp to plausible range

    # Sensitivity bracket: α_h ∈ {0.2, 0.4, 0.6} centered on the 0.4 default.
    # Spans seasonality-dominant (α_h=0.2) ↔ holiday-dominant (α_h=0.6) ↔
    # balanced (α_h=0.4). Reviewer can see how the chosen weight matters in
    # absolute terms — typical spread is small (<0.05) when both signals agree.
    sensitivity = {}
    for alpha_h in [0.2, 0.4, 0.6]:
        alpha_s = 1.0 - alpha_h
        m = alpha_h * h["holiday_multiplier"] + alpha_s * s["seasonality_multiplier"]
        label = f"alpha_h={alpha_h}" + (" (default)" if alpha_h == ALPHA_H else "")
        sensitivity[label] = round(max(0.5, min(1.8, m)), 3)

    return {
        "category": category,
        "country": country,
        "applicable": True,
        "demand_multiplier": round(demand_mult, 3),
        "formula": f"demand = {ALPHA_H}*holiday_mult + {ALPHA_S}*seasonality_mult",
        "weights": {"alpha_holiday": ALPHA_H, "alpha_seasonal": ALPHA_S},
        "holiday": h,
        "seasonality": s,
        "sensitivity_analysis": sensitivity,
        "rationale": (f"holiday={h['holiday_multiplier']} & "
                      f"seasonal={s['seasonality_multiplier']} "
                      f"-> demand={round(demand_mult, 3)}"),
    }


### Cell 5 — Tool 4 (`priceiq_weather.py`) · OpenWeather 5-day forecast

In [ ]:
%%writefile priceiq_weather.py
"""PriceIQ Tool 4 — get_weather_signal (OpenWeather 5-day forecast).

Conditional invocation: only sports / garden categories trigger the live API
call. Other categories short-circuit to `applicable=False, multiplier=1.0`
without spending API quota — Planner v2 enforces this in its tool sequence.

Multiplier formula (clamped to [0.80, 1.15]):
  mult = 1.0 - 0.30 * rain_prob_5d
       + 0.05  if 20 ≤ avg_temp ≤ 28 (mild)
       - 0.10  if avg_temp < 15 or > 32 (extreme)

Sample: 5 BR cities (Sao Paulo / Rio / Brasilia / Salvador / Fortaleza) × 5 days
× 3-hour intervals = 200 forecast points averaged per call.

Graceful degradation: API failure → `degraded=True, multiplier=1.0`. Never raises.

Public API:
    get_weather_signal(category, region='BR', api_key=None) -> dict

Token (OPENWEATHER_API_KEY, 32-hex-char):
    Colab → userdata.get('OPENWEATHER_API_KEY')
    Local → export OPENWEATHER_API_KEY=...
    Explicit → get_weather_signal('sports', api_key=key)
"""

import os
import requests
from datetime import datetime, timezone

# Top-5 BR metros by Olist order volume — covers Southeast (SP, RJ),
# Center-West (Brasília), and Northeast (Salvador, Fortaleza). Together
# they account for roughly 70% of Olist's delivered orders. 5-city sample
# is a free-tier-friendly compromise (free OpenWeather quota = 60 calls/min;
# 5 cities × 1 query = 5 calls per agent invocation, well under).
_BR_CITIES = [
    ("Sao Paulo",  -23.5505, -46.6333),
    ("Rio",        -22.9068, -43.1729),
    ("Brasilia",   -15.7801, -47.9292),
    ("Salvador",   -12.9714, -38.5014),
    ("Fortaleza",   -3.7172, -38.5434),
]

# Only these categories are weather-sensitive (e-commerce purchase decisions
# correlate with outdoor weather). Both Olist Portuguese names and common
# English aliases accepted, so the Planner can use either form.
# Other categories (eletrônicos / cama_mesa_banho / etc.) short-circuit
# to applicable=False, weather_multiplier=1.0 — saves OpenWeather API quota.
_WEATHER_SENSITIVE = {
    "esporte_lazer", "ferramentas_jardim",
    "sports", "garden", "esporte", "jardim",
}

_API = "https://api.openweathermap.org/data/2.5/forecast"
_BR_AVG_TEMP = 24.0  # Brazil annual mean (°C) — averaging tropical NE coast (~27°C)
                     # with subtropical SE (~21°C). Used only for `temp_anomaly`
                     # field; multiplier itself uses absolute temp bands.


def get_weather_signal(category: str, region: str = "BR", api_key: str = None) -> dict:
    """返回天气信号 dict，含 weather_multiplier ∈ [0.80, 1.15]。

    非天气敏感品类返回 applicable=False, multiplier=1.0；
    API 失败返回 degraded=True, multiplier=1.0（不影响主流程）。
    """
    if category.lower() not in _WEATHER_SENSITIVE:
        return _neutral(category, region, applicable=False,
                        rationale=f"'{category}' not weather-sensitive")

    if region != "BR":
        return _neutral(category, region, degraded=True,
                        rationale="Only BR supported in MVP")

    key = api_key or os.environ.get("OPENWEATHER_API_KEY")
    if not key:
        return _neutral(category, region, degraded=True,
                        rationale="No API key provided")

    rain_probs, temps = [], []
    for name, lat, lon in _BR_CITIES:
        try:
            r = requests.get(_API, params={
                "lat": lat, "lon": lon, "appid": key,
                "units": "metric", "cnt": 40,  # 40 forecast points × 3h = 5-day horizon
            }, timeout=8)
            r.raise_for_status()
            for entry in r.json().get("list", []):
                rain_probs.append(entry.get("pop", 0.0))
                temps.append(entry["main"]["temp"])
        except (requests.RequestException, KeyError, ValueError):
            continue  # 单城市失败跳过

    if not temps:
        return _neutral(category, region, degraded=True,
                        rationale="All forecast queries failed")

    rain_prob = sum(rain_probs) / len(rain_probs)
    avg_temp = sum(temps) / len(temps)
    temp_anomaly = avg_temp - _BR_AVG_TEMP

    # ── Multiplier: start 1.0, rain pulls down, extreme temp pulls down, mild adds ──
    # Coefficients are heuristics (no calibration target available in scope).
    # Clamped to [0.80, 1.15] so weather alone can't dominate the elasticity signal.
    mult = 1.0 - rain_prob * 0.30          # up to -0.30 at 100% rain
    if 20 <= avg_temp <= 28:
        mult += 0.05                       # mild outdoor weather → slight boost
    elif avg_temp < 15 or avg_temp > 32:
        mult -= 0.10                       # too cold or too hot → discouraging
    mult = max(0.80, min(1.15, mult))

    return {
        "category": category,
        "region": region,
        "applicable": True,
        "cities_sampled": len(_BR_CITIES),
        "rain_prob_5d": round(rain_prob, 3),
        "avg_temp_5d_c": round(avg_temp, 2),
        "temp_anomaly": round(temp_anomaly, 2),
        "weather_multiplier": round(mult, 3),
        "rationale": _rationale(rain_prob, avg_temp, mult),
        "fetched_at": datetime.now(timezone.utc).isoformat(),
        "degraded": False,
    }


def _neutral(category, region, applicable=True, degraded=False, rationale=""):
    """非敏感品类 / 降级 → 返回中性 multiplier=1.0。"""
    return {
        "category": category, "region": region,
        "applicable": applicable, "degraded": degraded,
        "weather_multiplier": 1.0, "rationale": rationale,
    }


def _rationale(rain_prob, avg_temp, mult):
    """Compose a human-readable weather rationale string for the agent's answer."""
    rain = "high rain" if rain_prob > 0.5 else \
           "moderate rain" if rain_prob > 0.3 else "low rain"
    temp = "hot" if avg_temp > 30 else \
           "mild" if avg_temp > 20 else "cool"
    direction = "favorable" if mult > 1.02 else \
                "unfavorable" if mult < 0.95 else "neutral"
    return f"{rain}, {temp} — {direction} for outdoor categories"


if __name__ == "__main__":
    # 自测：key 从环境变量读，不进代码
    if not os.environ.get("OPENWEATHER_API_KEY"):
        print("⚠️  OPENWEATHER_API_KEY not set. Run:")
        print("    export OPENWEATHER_API_KEY=your_key && python priceiq_weather.py")
        exit(0)

    for cat in ["esporte_lazer", "ferramentas_jardim", "eletronicos"]:
        print(f"\n── {cat} ──")
        for k, v in get_weather_signal(cat).items():
            print(f"  {k}: {v}")


### Cell 6 — Tool 5 (`priceiq_simulator.py`) · 3-scenario revenue projection

In [ ]:
%%writefile priceiq_simulator.py
"""PriceIQ Tool 5 — simulate_revenue_impact (3-scenario CI propagation).

Combines elasticity β + demand_multiplier + weather_multiplier into a
forward simulation of a proposed price change.

Formula:
    new_qty     = current_qty × (1+Δp)^β × demand_mult × weather_mult
    new_revenue = new_qty × current_price × (1+Δp)
    Δrevenue%   = new_revenue / current_revenue − 1

Three scenarios from β's 95% CI:
    pessimistic_beta_low · central · optimistic_beta_high

Public API:
    simulate_revenue_impact(category, price_change_pct,
                            elasticity=None, demand_signal=None,
                            weather_signal=None) -> dict
        If upstream signals are omitted, the simulator self-fetches them.
        Returns scenarios + formula + caveat (complementary to elasticity's caveat).
"""

from typing import Optional


def simulate_revenue_impact(
    category: str,
    price_change_pct: float,
    elasticity: Optional[dict] = None,
    demand_signal: Optional[dict] = None,
    weather_signal: Optional[dict] = None,
) -> dict:
    """模拟价格变动 Δp 对收益的影响（含 3 场景）。

    Args:
        category: 葡语品类名
        price_change_pct: 价格变动比例（-0.10 = 降价 10%）。
            合理范围 [-0.5, +0.5]；超过会触发数学奇点（dp=-1 时 (1+dp)^β=0^β
            div-by-zero for β<0）或不可信外推（dp=-0.9 + β=-3 → factor=1000）。
            超界时返回 success=False。
        elasticity / demand_signal / weather_signal: 上游工具输出，
            未提供时自动调用对应 Tool。

    Returns:
        含 current / new_price / scenarios（3 个）+ formula + caveat 的 dict。
    """
    # Validate input range — refuse extreme deltas to avoid math singularities
    if not (-0.5 <= price_change_pct <= 0.5):
        return {"category": category, "success": False,
                "message": (f"price_change_pct={price_change_pct} outside [-0.5, +0.5]. "
                            "Extreme price changes produce non-credible projections; "
                            "consider running multiple smaller deltas instead.")}

    if elasticity is None:
        from priceiq_elasticity import calculate_price_elasticity
        elasticity = calculate_price_elasticity(category)
    if not elasticity.get("success"):
        return {"category": category, "success": False,
                "message": f"Elasticity unavailable: {elasticity.get('message','?')}"}

    if demand_signal is None:
        from priceiq_demand import get_demand_signals
        demand_signal = get_demand_signals(category)
    if weather_signal is None:
        from priceiq_weather import get_weather_signal
        weather_signal = get_weather_signal(category)

    # Pull baseline state (price + 3-month-avg quantity) from Tool 1
    from priceiq_data import query_sales_data
    sales = query_sales_data(category)
    if not sales.get("found"):
        return {"category": category, "success": False, "message": "No sales data"}

    current_price = sales["price_stats"]["mean"]
    monthly = [m["n_orders"] for m in sales["monthly_panel"][-3:]]   # last 3 months
    if not monthly:
        return {"category": category, "success": False,
                "message": "Empty monthly_panel — cannot estimate current quantity"}
    current_qty = sum(monthly) / len(monthly)
    current_revenue = current_qty * current_price

    # CI bounds drive the pessimistic / optimistic scenarios; β_central is the
    # recommended point estimate (already accounts for multicollinearity fallback).
    beta_central = elasticity["elasticity_beta"]
    ci = elasticity.get("ci_95", [beta_central, beta_central])
    beta_lo, beta_hi = ci

    # Default 1.0 = neutral (no signal). This is the documented degraded
    # behavior (ADR-004): if Tool 3 / Tool 4 returned partial output, the
    # simulator still produces a directional answer rather than aborting.
    demand_mult = demand_signal.get("demand_multiplier", 1.0)
    weather_mult = weather_signal.get("weather_multiplier", 1.0)

    delta_p = price_change_pct
    new_price = current_price * (1 + delta_p)

    def _scenario(beta):
        """Compute one revenue scenario for a given β; closes over current_*, mults, delta_p."""
        qty_factor = (1 + delta_p) ** beta * demand_mult * weather_mult
        new_qty = current_qty * qty_factor
        new_revenue = new_qty * new_price
        # Rounding convention: β to 3dp (statistical precision), qty to 1dp
        # (whole-unit-ish), revenue to 2dp ($cents), pct change to 4dp (4 sig fig).
        return {
            "beta_used": round(beta, 3),
            "new_qty_monthly": round(new_qty, 1),
            "new_revenue_monthly": round(new_revenue, 2),
            "revenue_change_pct": round(new_revenue / current_revenue - 1, 4),
            "qty_change_pct": round(qty_factor - 1, 4),
        }

    return {
        "category": category,
        "success": True,
        "price_change_pct": price_change_pct,
        "current": {
            "price": round(current_price, 2),
            "monthly_qty_avg_3mo": round(current_qty, 1),
            "monthly_revenue_avg_3mo": round(current_revenue, 2),
        },
        "new_price": round(new_price, 2),
        "elasticity_beta_central": beta_central,
        "elasticity_ci_95": ci,
        "demand_multiplier": demand_mult,
        "weather_multiplier": weather_mult,
        # Keys are named after which β CI end is used (statistics POV). The
        # *revenue* ordering of these three is NOT guaranteed — for an elastic
        # good with a price decrease, β_low (most elastic) gives the *highest*
        # revenue. The Streamlit UI re-sorts by revenue and labels with
        # user-facing Pessimistic / Central / Optimistic. Don't rely on the
        # dict-key order to mean "low → high revenue".
        "scenarios": {
            "pessimistic_beta_low":  _scenario(beta_lo),
            "central":               _scenario(beta_central),
            "optimistic_beta_high":  _scenario(beta_hi),
        },
        "formula": "new_qty = current_qty * (1+dp)^beta * demand_mult * weather_mult",
        "caveat": (
            "Forward simulation based on log-log elasticity. Results are "
            "DIRECTIONAL not CAUSAL predictions. CI bounds reflect statistical "
            "uncertainty in beta only; demand/weather multiplier uncertainty is "
            "not propagated. Use alongside elasticity tool's causal_caveat."
        ),
    }


### Cell 7 — Agent (`priceiq_agent.py`) · TOOLS schema + Planner v1/v2 + Executor loop

In [ ]:
%%writefile priceiq_agent.py
"""PriceIQ Agent — Track B (manual tool_use orchestration).

Pipeline:
    User query
       ↓
    Planner (claude-sonnet-4-5)  — XML few-shot prompt v2 with 71-category context
       ↓
    JSON plan {category_pt, tool_sequence, user_intent}
       ↓
    Executor (claude-haiku-4-5)  — manual tool_use loop, MAX_ITER=8 kill-switch
       ↓
    Final answer with verbatim causal_caveat

Hard limits / FinOps:
  MAX_ITERATIONS         = 8     (kill-switch, ADR-003)
  MEMORY_THRESHOLD_CHARS = 30000 (compress history when exceeded; v1 was 8K, see F-01)
  Telemetry              every tool call's input/output/tokens/latency logged

Public API:
    priceiq_agent(user_query, anthropic_client, verbose=True,
                  planner_version='v2') -> dict
        planner_version='v1' reproduces the Shortcut Bias demo failure.
        Returns {success, answer, plan, telemetry}.

Constants:
    TOOLS              — 5-element JSON schema list (Anthropic tool_use format)
    PLANNER_PROMPT_V1  — under-specified (demo-failure prompt)
    PLANNER_PROMPT_V2  — production prompt with 71 categories + 5 few-shots
    EXECUTOR_PROMPT    — synthesizes final answer with verbatim causal_caveat
"""

import json
import time
from typing import Optional


# ── Hard limits ────────────────────────────────────────────────
MAX_ITERATIONS = 8
MEMORY_THRESHOLD_CHARS = 30000   # 5-tool 场景下一般 8-12K，提高阈值避免误触
PLANNER_MODEL = "claude-sonnet-4-5"
EXECUTOR_MODEL = "claude-haiku-4-5"
JUDGE_MODEL = "claude-sonnet-4-5"


# ── TOOLS JSON schema (Anthropic tool_use format) ──────────────
TOOLS = [
    {
        "name": "query_sales_data",
        "description": (
            "Query historical Olist e-commerce sales data for a category. "
            "Returns price stats, freight, and monthly_panel needed for elasticity."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {
                    "type": "string",
                    "description": "Olist Portuguese category name (e.g., 'esporte_lazer', 'ferramentas_jardim').",
                },
                "start_date": {"type": "string", "description": "ISO YYYY-MM-DD; optional"},
                "end_date": {"type": "string", "description": "ISO YYYY-MM-DD; optional"},
            },
            "required": ["category"],
        },
    },
    {
        "name": "calculate_price_elasticity",
        "description": (
            "Estimate price elasticity beta via log-log OLS with freight control + "
            "multicollinearity diagnostics. Returns recommended_beta with fallback to "
            "naive when controlled model is unstable. Includes causal_caveat."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string"},
                "start_date": {"type": "string"},
                "end_date": {"type": "string"},
            },
            "required": ["category"],
        },
    },
    {
        "name": "get_demand_signals",
        "description": (
            "Combine BR holiday proximity + Olist historical seasonality into a "
            "demand_multiplier. Includes explicit formula and sensitivity_analysis "
            "across alpha_holiday weights."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string"},
                "country": {"type": "string", "description": "Default 'BR'"},
            },
            "required": ["category"],
        },
    },
    {
        "name": "get_weather_signal",
        "description": (
            "Fetch real-time 5-day OpenWeather forecast for 5 BR cities, returning "
            "weather_multiplier in [0.80, 1.15]. Only meaningful for sports/garden "
            "categories; auto short-circuits for non-weather-sensitive ones."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string"},
                "region": {"type": "string", "description": "Default 'BR'"},
            },
            "required": ["category"],
        },
    },
    {
        "name": "simulate_revenue_impact",
        "description": (
            "Simulate revenue change under a proposed price_change_pct, combining "
            "elasticity beta + demand_mult + weather_mult. Outputs 3 scenarios "
            "(pessimistic/central/optimistic) based on beta's 95% CI."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "category": {"type": "string"},
                "price_change_pct": {
                    "type": "number",
                    "description": "Decimal (-0.10 = -10%, +0.15 = +15%). Range [-0.5, +0.5]; "
                                   "extreme values are refused to avoid math singularities.",
                },
            },
            "required": ["category", "price_change_pct"],
        },
    },
]


def _tool_dispatch(name: str, args: dict) -> dict:
    """把 tool name 派发到本地函数。"""
    if name == "query_sales_data":
        from priceiq_data import query_sales_data
        return query_sales_data(**args)
    if name == "calculate_price_elasticity":
        from priceiq_elasticity import calculate_price_elasticity
        return calculate_price_elasticity(**args)
    if name == "get_demand_signals":
        from priceiq_demand import get_demand_signals
        return get_demand_signals(**args)
    if name == "get_weather_signal":
        from priceiq_weather import get_weather_signal
        return get_weather_signal(**args)
    if name == "simulate_revenue_impact":
        from priceiq_simulator import simulate_revenue_impact
        return simulate_revenue_impact(**args)
    return {"error": f"Unknown tool: {name}"}


# ── Planner Prompt v1 (DELIBERATELY UNDER-SPECIFIED — for PVC log demo) ──
# v1 故意不提供完整 71 类的葡语映射，让 Planner 经常 hallucinate 品类名（如 sports → informatica_acessorios）。
# 这是 demo 视频要剖析的"架构失败"。
PLANNER_PROMPT_V1 = """<context>
You are PriceIQ Planner, an AI that decomposes pricing questions into a tool-call plan.
You have access to 5 tools that work on Brazilian Olist e-commerce data.
</context>

<task>
Given the user's pricing question, output a JSON plan listing which tools to call and in what order.
Tools available: query_sales_data, calculate_price_elasticity, get_demand_signals,
get_weather_signal, simulate_revenue_impact.
</task>

<rules>
- Always start with query_sales_data to confirm category exists.
- Then calculate_price_elasticity.
- Then if a price change is mentioned, simulate_revenue_impact.
- Map English category names to Olist Portuguese names yourself.
</rules>

<output_format>
Output ONLY a JSON object:
{
  "category_pt": "<Olist portuguese category>",
  "tool_sequence": ["tool1", "tool2", ...],
  "user_intent": "<one sentence>"
}
</output_format>"""


# ── Planner Prompt v2 (FIXED — adds full category list + few-shot) ────────
# v2 加入完整 71 类 + 5 个 few-shot（含 sports / garden 显式映射）。
PLANNER_PROMPT_V2 = """<context>
You are PriceIQ Planner, an AI that decomposes pricing questions into a tool-call plan.
You have access to 5 tools on Brazilian Olist e-commerce data.

Olist has exactly 71 product categories (Portuguese names). The most common are:
- cama_mesa_banho (bed_bath_table)
- beleza_saude (health_beauty)
- esporte_lazer (sports_leisure)              ← USE FOR ANY "SPORTS" QUERY
- moveis_decoracao (furniture_decor)
- informatica_acessorios (computers_accessories)  ← computers/laptops only
- utilidades_domesticas (housewares)
- relogios_presentes (watches_gifts)
- telefonia (telephony)                       ← phones only
- ferramentas_jardim (garden_tools)           ← USE FOR ANY "GARDEN" QUERY
- automotivo (auto)
- eletronicos (electronics)                   ← TVs/audio only, NOT sports/computers
</context>

<task>
Given the user's pricing question, output a JSON plan listing which tools to call.
</task>

<examples>
Q: "Should we discount sports gear next month?"
A: {"category_pt": "esporte_lazer", "tool_sequence": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "get_weather_signal", "simulate_revenue_impact"], "user_intent": "Evaluate impact of discounting sports category"}

Q: "What about a 15% price hike on garden tools?"
A: {"category_pt": "ferramentas_jardim", "tool_sequence": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "get_weather_signal", "simulate_revenue_impact"], "user_intent": "Simulate +15% price change on garden tools"}

Q: "How elastic are bed sheets?"
A: {"category_pt": "cama_mesa_banho", "tool_sequence": ["query_sales_data", "calculate_price_elasticity"], "user_intent": "Inquiry about bed_bath_table elasticity (no price change to simulate)"}

Q: "Should we drop laptop prices?"
A: {"category_pt": "informatica_acessorios", "tool_sequence": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"], "user_intent": "Computer/laptop discount evaluation; weather not relevant"}

Q: "How much should I cut TV prices to clear inventory?"
A: {"category_pt": "eletronicos", "tool_sequence": ["query_sales_data", "calculate_price_elasticity", "get_demand_signals", "simulate_revenue_impact"], "user_intent": "TV/electronics price cut to clear stock"}
</examples>

<rules>
- "sports" / "athletic" / "gym" → esporte_lazer (NOT informatica_acessorios)
- "garden" / "plants" / "tools" → ferramentas_jardim
- weather tool ONLY for sports/garden; skip for others
- Always end with simulate_revenue_impact when a specific Δprice is mentioned
</rules>

<output_format>
Output ONLY a JSON object matching the example schema. No prose.
</output_format>"""


# Default to v2 (production); switch to v1 to reproduce the demo failure
PLANNER_SYSTEM_PROMPT = PLANNER_PROMPT_V2


# ── Executor system prompt ─────────────────────────────────────
EXECUTOR_PROMPT = """<context>
You are PriceIQ Executor. You have a plan from the Planner and 5 tools.
Execute the plan step-by-step using tool_use, gathering structured data.
You have a hard limit of 8 iterations — finalize promptly and do not loop.
</context>

<rules>
- Call tools in the order specified by the plan.
- Pass category_pt (Portuguese) as the category argument to all tools.
- After all tools complete, synthesize a final recommendation that includes:
  1. The recommended beta and its confidence interval
  2. The 3-scenario revenue projection (if simulate was called)
  3. The causal_caveat verbatim from the elasticity tool
  4. The multicollinearity_warning if True
- If a tool returns success=False / found=False, report the failure in your
  final answer and move on. Do not retry the same tool — neither with
  identical nor with variant arguments. Each tool is called at most once.
- Be concise; use bullet points; no prose padding.
</rules>"""


# ── Memory compression ─────────────────────────────────────────
def _compress_memory(messages: list, anthropic_client) -> list:
    """history 太长时，summarize 工具调用历史。"""
    # Anthropic responses contain TextBlock / ToolUseBlock objects (not plain dicts).
    # `default=str` makes json.dumps fall back to repr() for those — see F-04.
    try:
        serialized = json.dumps(messages, default=str)
    except Exception:
        serialized = str(messages)
    if len(serialized) < MEMORY_THRESHOLD_CHARS:
        return messages

    # Compress: ask Haiku to summarize the entire history into 3-5 bullets.
    # Truncate input to 2× the threshold to bound the compress call cost.
    summary = anthropic_client.messages.create(
        model=EXECUTOR_MODEL,
        max_tokens=512,
        system="Summarize this PriceIQ tool execution history into 3-5 bullets focused on key findings (beta, demand_mult, weather_mult, recommended_action). Drop redundant raw data.",
        messages=[{"role": "user", "content": serialized[:MEMORY_THRESHOLD_CHARS * 2]}],
    ).content[0].text

    # Replace history with single user message containing the summary + an
    # explicit "finalize NOW" instruction to prevent role confusion (F-01).
    return [{"role": "user", "content": (
        f"<memory_summary>\n{summary}\n</memory_summary>\n\n"
        "Based on the above tool results summarized in memory, generate the FINAL "
        "pricing recommendation NOW. Do NOT ask follow-up questions, do NOT call "
        "more tools. Synthesize the recommendation with: (1) recommended beta + CI, "
        "(2) 3-scenario revenue projection, (3) causal_caveat, (4) multicollinearity "
        "warning if any."
    )}]


# ── Main agent entry point ─────────────────────────────────────
def priceiq_agent(user_query: str, anthropic_client, verbose: bool = True,
                  planner_version: str = "v2") -> dict:
    """Run the full Planner → Executor pipeline.

    Args:
        user_query:        Natural-language pricing question.
        anthropic_client:  `anthropic.Anthropic()` instance.
        verbose:           Print iteration progress to stdout.
        planner_version:   'v2' (production) | 'v1' (Shortcut Bias demo).

    Returns:
        {
          "success":   bool,
          "answer":    str,            # final Executor synthesis
          "plan":      dict,           # {category_pt, tool_sequence, user_intent}
          "telemetry": {
              "query":            str,
              "planner_version":  str,
              "planner_tokens":   {input, output},
              "executor_tokens":  [{iter, input, output}, …],
              "tool_calls":       [{iter, tool, input, latency_s, error, result_keys}, …],
              "errors":           [{iter, tool, err}, …],   # per-tool errors during execution
              "iterations":       int,
              "latency_s":        float,
              "started_at":       float,
              "finished_at":      float,
              "plan":             dict,                      # echo of Planner output
              "final_answer":     str,                       # only when success=True
              "error":            str,                       # only when success=False (e.g. MAX_ITER hit)
          }
        }
    """
    prompt = PLANNER_PROMPT_V1 if planner_version == "v1" else PLANNER_PROMPT_V2
    telemetry = {"query": user_query, "planner_version": planner_version,
                 "started_at": time.time(), "tool_calls": [], "errors": []}

    # ── Planner ────────────────────────────────────────────
    if verbose:
        print(f"\n[Planner {planner_version}] {PLANNER_MODEL}")
    plan_resp = anthropic_client.messages.create(
        model=PLANNER_MODEL,
        max_tokens=512,
        system=prompt,
        messages=[{"role": "user", "content": user_query}],
    )
    plan_text = plan_resp.content[0].text.strip()
    telemetry["planner_tokens"] = {
        "input": plan_resp.usage.input_tokens,
        "output": plan_resp.usage.output_tokens,
    }
    try:
        plan = json.loads(plan_text)
    except json.JSONDecodeError:
        # 抽取第一个 {...}
        import re
        m = re.search(r"\{.*\}", plan_text, re.DOTALL)
        plan = json.loads(m.group(0)) if m else {"error": "plan_parse_failed", "raw": plan_text}
    if verbose:
        print(f"[Plan] {plan}")
    telemetry["plan"] = plan

    if "category_pt" not in plan:
        return {"success": False, "telemetry": telemetry,
                "error": "Planner failed to produce category_pt"}

    # ── Executor (manual tool_use loop) ─────────────────────
    if verbose:
        print(f"\n[Executor] {EXECUTOR_MODEL}")

    enriched_query = (
        f"User question: {user_query}\n\n"
        f"Plan from Planner:\n{json.dumps(plan, indent=2)}\n\n"
        f"Execute the tool_sequence in order. Use category_pt='{plan['category_pt']}' for all category arguments."
    )
    messages = [{"role": "user", "content": enriched_query}]

    for iteration in range(MAX_ITERATIONS):
        if verbose:
            print(f"  [iter {iteration+1}/{MAX_ITERATIONS}]")

        # Memory 压缩
        messages = _compress_memory(messages, anthropic_client)

        resp = anthropic_client.messages.create(
            model=EXECUTOR_MODEL,
            max_tokens=2048,
            system=EXECUTOR_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        telemetry.setdefault("executor_tokens", []).append({
            "iter": iteration + 1,
            "input": resp.usage.input_tokens,
            "output": resp.usage.output_tokens,
        })

        # 把 assistant 回应加进 history
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            # Final answer
            final_text = "".join(b.text for b in resp.content if hasattr(b, "text"))
            telemetry["final_answer"] = final_text
            telemetry["finished_at"] = time.time()
            telemetry["latency_s"] = telemetry["finished_at"] - telemetry["started_at"]
            telemetry["iterations"] = iteration + 1
            return {"success": True, "answer": final_text, "plan": plan, "telemetry": telemetry}

        if resp.stop_reason == "tool_use":
            tool_results = []
            for block in resp.content:
                if block.type == "tool_use":
                    name, inp = block.name, block.input
                    if verbose:
                        print(f"    [tool_use] {name}({inp})")
                    t0 = time.time()
                    try:
                        result = _tool_dispatch(name, inp)
                        err = None
                    except Exception as e:
                        result = {"error": str(e)}
                        err = str(e)
                        telemetry["errors"].append({"iter": iteration+1, "tool": name, "err": err})
                    latency = time.time() - t0
                    telemetry["tool_calls"].append({
                        "iter": iteration + 1,
                        "tool": name,
                        "input": inp,
                        "latency_s": round(latency, 3),
                        "error": err,
                        "result_keys": list(result.keys()) if isinstance(result, dict) else None,
                    })
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)[:8000],
                    })
            messages.append({"role": "user", "content": tool_results})

    # MAX_ITER hit
    telemetry["error"] = f"Hit MAX_ITERATIONS={MAX_ITERATIONS} without end_turn"
    telemetry["finished_at"] = time.time()
    return {"success": False, "telemetry": telemetry,
            "answer": "Agent exceeded iteration cap; partial results in telemetry."}


## Cell 8 — End-to-end demo

In [ ]:
import importlib, sys
for m in ['priceiq_data','priceiq_elasticity','priceiq_demand',
          'priceiq_weather','priceiq_simulator','priceiq_agent']:
    if m in sys.modules: importlib.reload(sys.modules[m])

import anthropic
from priceiq_agent import priceiq_agent
client = anthropic.Anthropic()

result = priceiq_agent(
    'Should we discount garden tools by 10% next month?',
    client, verbose=True, planner_version='v2',
)
print('\n=== Final answer ===')
print(result['answer'])


## Cell 9 — Optional: 50-case evaluation

**Cost**: ~$2.57 (50 agent calls × $0.029 + 50 judge calls × $0.005 + 30 consistency runs × $0.029). **Wall-clock**: ~25 min.

Uncomment to run.

In [ ]:
# %%writefile eval_suite.py
# (paste eval_suite.py contents here, then run the next cell)
# from eval_suite import run_full_eval
# results = run_full_eval(client, priceiq_agent, n=50, consistency_runs=3)
# import json; print(json.dumps(results['summary'], indent=2))


## Cell 10 — Telemetry export

In [ ]:
import json, pathlib
out = pathlib.Path('telemetry.json')
out.write_text(json.dumps(result['telemetry'], indent=2, default=str))
print(f'Saved telemetry to {out.absolute()}')
